In [1]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import re
import numpy as np
import pandas as pd

import nltk
nltk.download('punkt_tab')
from nltk.tokenize import sent_tokenize

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\PC\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
N_dovish_V_neg = ['inflation expectation', 'interest rate',
                  'bank rate', 'fund rate',
             'price', 'economic activity', 
             'inflation', ' employment']

N_dovish_V_pos = ['unemployment', 'growth',
                  'monetary policy', 'exchange rate',
                  'productivity', 'demand', 'deficit', 'job market']

N_dovish_V_neg_ECB = N_dovish_V_neg + [
    'deposit facility rate', 'refinancing rate', 'key ecb interest rate',
    'lending facility', 'spread', 'financing condition']

N_dovish_V_pos_ECB = N_dovish_V_pos + [
    'asset purchase', 'monetary accommodation', 'reinvestment',
    'fragmentation']


V_neg = ['anchor', 'target', 'cut', 'subdue',
         'declin', 'decreas', 'reduc', 
         'low', 'drop', 'tighten',
         'fall', 'fell', 'decelerat', 'slow',
         'pause', 'pausing',
         'stable', 'nonaccelerating', 'downward']

V_pos = ['rise', 'rising', 'increas', 'expand', 'improv',
         'strong', 'upward', 'rais', 'high', 'rapid', 'ease', 'easing']

negation = ["weren't", 'were not', 'was not', "wasn't",
            'did not', "didn't", 'do not', "don't",
            'will not', "won't"]


In [3]:
def search_and_count(all_content, N_neg=None, N_pos=None):
    N_neg = N_neg if N_neg is not None else N_dovish_V_neg
    N_pos = N_pos if N_pos is not None else N_dovish_V_pos
    dovish, hawkish = [], []
    for text in all_content['text'].tolist():
        count_d = count_h = 0
        for sent_all in sent_tokenize(str(text)):
            sent_all = sent_all.lower().replace(';', ',').replace('\t', ' ').strip()
            for sent in sent_all.split(','):
                n_neg = any(w in sent for w in N_neg)
                n_pos = any(w in sent for w in N_pos)
                v_neg = any(w in sent for w in V_neg)
                v_pos = any(w in sent for w in V_pos)
                neg   = any(w in sent for w in negation)
                if (n_neg and v_neg and not neg) or (n_pos and v_pos and not neg): count_d += 1
                if (n_neg and v_neg and neg)     or (n_pos and v_pos and neg):     count_h += 1
                if (n_neg and v_pos and not neg) or (n_pos and v_neg and not neg): count_h += 1
                if (n_neg and v_pos and neg)     or (n_pos and v_neg and neg):     count_d += 1
        dovish.append(count_d); hawkish.append(count_h)
    out = all_content.copy()
    out['hawkish'] = hawkish
    out['dovish']  = dovish
    return out.drop(columns=['text'])

In [4]:
input_file  = 'C:/Users/PC/Desktop/TFM/BIG DATA/TEXT SENTIMENT/data/ECB_TEXT (2013-2024)_v2.xlsx'
output_file = 'C:/Users/PC/Desktop/TFM/BIG DATA/TEXT SENTIMENT/data/output files/output_search_count_ECB_v2.xlsx'

In [5]:
# remarks - por parrafo   (PRIMERA: mode='w')
df = pd.read_excel(input_file, sheet_name='remarks_para', engine="openpyxl")
out = search_and_count(df)
with pd.ExcelWriter(output_file, mode='w', engine="openpyxl") as w:
    out.to_excel(w, sheet_name='remarks_count', index=False)

In [6]:
# answers - por parrafo   (SEGUNDA: mode='a')
df = pd.read_excel(input_file, sheet_name='answers_para', engine="openpyxl")
out = search_and_count(df)
with pd.ExcelWriter(output_file, mode='a', engine="openpyxl") as w:
    out.to_excel(w, sheet_name='answers_count', index=False)